# Healthcare Analytics for Doctor Visits
## Complete Python Data Analytics Project

**Goal:** Explore patterns associated with doctor visits and prepare a clean dataset for an interactive Power BI dashboard.

**Dataset:** Healthcare Analytics for Doctor Visits.csv  
**Rows:** 5,190 | **Original columns:** 13 | **Analytical columns after removing row index:** 12

### Important interpretation notes
- This is observational data. Associations do not establish causation.
- `age` is stored as a decimal (0.19–0.72) and `income` as a numeric scale (0–1.5). Verify units from the source documentation before converting them into years or currency.
- Binary text fields (`private`, `freepoor`, `freerepat`, `nchronic`, `lchronic`) are retained as provided. Verify their exact definitions from the data dictionary before making policy conclusions.
- `Unnamed: 0` appears to be a row identifier, so it is removed from analysis.


## 1. Import libraries

Pandas reads and manipulates data; NumPy provides numerical tools; Matplotlib and Seaborn create charts. If any library is missing, run `pip install pandas numpy matplotlib seaborn scipy` in your notebook environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the CSV

If you use Google Colab, upload the CSV in the Files panel. If you use Jupyter, place it beside this notebook or change the path below.

In [ ]:
file_path = "Healthcare Analytics for Doctor Visits.csv"
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
display(df.head())


## 3. Understand the structure

`shape` gives rows and columns. `info()` shows data types and non-null counts. `describe()` summarizes numerical variables.

In [ ]:
print("Column names:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)


## 4. Data quality checks

Check missing values, exact duplicate rows, and category values before cleaning.

In [ ]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())

for col in df.select_dtypes(include="object").columns:
    print(f"\n{col} values:")
    print(df[col].value_counts(dropna=False))


## 5. Clean the data

We preserve the original dataframe and create a working copy. The row-index column is dropped because it appears to be a serial number rather than an analytical measure. No missing values or exact duplicate rows were found in the uploaded file.

In [ ]:
data = df.copy()
data = data.drop(columns=["Unnamed: 0"], errors="ignore")

print("Cleaned shape:", data.shape)
display(data.head())


## 6. Summary of the target variable: `visits`

Actual results from the uploaded file:
- Total records: **5,190**
- Mean visits per record: **0.302**
- Median visits: **0**
- Maximum visits: **9**
- Zero-visit records: **4,141 (79.79%)**
- Records with one or more visits: **1,049 (20.21%)**

These percentages describe records within the dataset. Do not interpret them as people who never visit a doctor without knowing the survey period and collection method.

In [ ]:
print("Records:", len(data))
print("Mean visits:", round(data["visits"].mean(), 3))
print("Median visits:", data["visits"].median())
print("Maximum visits:", data["visits"].max())

visit_counts = data["visits"].value_counts().sort_index()
display(visit_counts.rename_axis("visits").to_frame("record_count"))

zero_count = int((data["visits"] == 0).sum())
positive_count = int((data["visits"] > 0).sum())
print(f"Zero visits: {zero_count} ({zero_count/len(data)*100:.2f}%)")
print(f"One or more visits: {positive_count} ({positive_count/len(data)*100:.2f}%)")


## 7. Visualize doctor-visit distribution

The expected shape is highly concentrated at zero. This helps explain why the mean is greater than the median.

In [ ]:
plt.figure(figsize=(9, 5))
sns.countplot(data=data, x="visits", color="steelblue")
plt.title("Distribution of Doctor Visits")
plt.xlabel("Number of Doctor Visits")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()


## 8. Gender comparison

Actual results:
- Female: 2,702 records; average visits 0.362; 24.43% have at least one visit.
- Male: 2,488 records; average visits 0.236; 15.63% have at least one visit.

This is a descriptive difference and does not explain its cause.

In [ ]:
gender_summary = data.groupby("gender").agg(
    records=("visits", "size"),
    average_visits=("visits", "mean"),
    median_visits=("visits", "median"),
    percent_with_visits=("visits", lambda s: (s > 0).mean() * 100)
).round(3)
display(gender_summary)

plt.figure(figsize=(7, 5))
sns.barplot(data=data, x="gender", y="visits", errorbar=None, color="steelblue")
plt.title("Average Doctor Visits by Gender")
plt.xlabel("Gender")
plt.ylabel("Average Visits")
plt.tight_layout()
plt.show()


## 9. Illness count and doctor visits

Actual group averages:
- 0 illnesses: 0.079 visits
- 1 illness: 0.295
- 2 illnesses: 0.404
- 3 illnesses: 0.421
- 4 illnesses: 0.577
- 5 illnesses: 0.814

The general pattern is positive: higher illness counts tend to coincide with more visits. It is not perfectly increasing at every step and does not establish causation.

In [ ]:
illness_summary = data.groupby("illness").agg(
    records=("visits", "size"),
    average_visits=("visits", "mean"),
    percent_with_visits=("visits", lambda s: (s > 0).mean() * 100)
).round(3)
display(illness_summary)

plt.figure(figsize=(8, 5))
sns.barplot(data=data, x="illness", y="visits", errorbar=None, color="steelblue")
plt.title("Illness Count vs Average Doctor Visits")
plt.xlabel("Reported Illness Count")
plt.ylabel("Average Visits")
plt.tight_layout()
plt.show()


## 10. Age and income: exploratory bands

The units of `age` and `income` are not confirmed here. Bands below are numeric ranges only, not real-world age or currency labels. Use the original data dictionary to confirm their meaning.

In [ ]:
data["age_value_group"] = pd.cut(
    data["age"], bins=[0, .2, .3, .4, .5, .6, .8],
    labels=["<=0.20", "0.20–0.30", "0.30–0.40", "0.40–0.50", "0.50–0.60", ">0.60"],
    include_lowest=True
)
data["income_value_group"] = pd.cut(
    data["income"], bins=[-0.001, .25, .5, .75, 1, 1.25, 1.500001],
    labels=["0–0.25", "0.25–0.50", "0.50–0.75", "0.75–1.00", "1.00–1.25", ">1.25"],
    include_lowest=True
)

age_summary = data.groupby("age_value_group", observed=False).agg(
    records=("visits", "size"), average_visits=("visits", "mean")
).round(3)
income_summary = data.groupby("income_value_group", observed=False).agg(
    records=("visits", "size"), average_visits=("visits", "mean")
).round(3)

display(age_summary)
display(income_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=data, x="age_value_group", y="visits", errorbar=None, ax=axes[0], color="steelblue")
axes[0].set_title("Average Visits by Age Value Band")
axes[0].set_xlabel("Age value band")
axes[0].set_ylabel("Average visits")
axes[0].tick_params(axis="x", rotation=35)

sns.barplot(data=data, x="income_value_group", y="visits", errorbar=None, ax=axes[1], color="darkorange")
axes[1].set_title("Average Visits by Income Value Band")
axes[1].set_xlabel("Income value band")
axes[1].set_ylabel("Average visits")
axes[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## 11. Health, reduced activity and chronic-condition indicators

For numeric measures, inspect group averages and distributions. For binary text fields, compare the categories as recorded, but confirm their exact definitions before interpreting them as clinical or insurance status.

In [ ]:
for col in ["health", "reduced"]:
    summary = data.groupby(col).agg(
        records=("visits", "size"),
        average_visits=("visits", "mean"),
        percent_with_visits=("visits", lambda s: (s > 0).mean() * 100)
    ).round(3)
    print(f"\nVisits grouped by {col}:")
    display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=data, x="health", y="visits", lowess=True,
            scatter_kws={"alpha": 0.18, "s": 18}, line_kws={"color": "darkorange"}, ax=axes[0])
axes[0].set_title("Visits and Poor-Health Days")
sns.regplot(data=data, x="reduced", y="visits", lowess=True,
            scatter_kws={"alpha": 0.18, "s": 18}, line_kws={"color": "darkorange"}, ax=axes[1])
axes[1].set_title("Visits and Reduced-Activity Days")
plt.tight_layout()
plt.show()

binary_cols = ["private", "freepoor", "freerepat", "nchronic", "lchronic"]
for col in binary_cols:
    summary = data.groupby(col).agg(
        records=("visits", "size"),
        average_visits=("visits", "mean"),
        percent_with_visits=("visits", lambda s: (s > 0).mean() * 100)
    ).round(3)
    print(f"\nVisits by {col} category:")
    display(summary)


## 12. Correlation analysis

Spearman correlation is useful for checking monotonic relationships, especially when variables are skewed or ordinal. Correlation is not causation.

In [ ]:
numeric_cols = ["visits", "age", "income", "illness", "reduced", "health"]
spearman_corr = data[numeric_cols].corr(method="spearman")
display(spearman_corr.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Spearman Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()


### Actual correlation findings
- Reduced-activity days vs visits: **+0.336**
- Illness count vs visits: **+0.263**
- Poor-health days vs visits: **+0.178**
- Age value vs visits: **+0.148**
- Income value vs visits: **−0.093**

These are weak-to-moderate associations. They are useful leads for further analysis, not causal effects.

## 13. Optional statistical test: compare visits by gender

Visit counts are highly skewed and have many zeros, so a Mann–Whitney U test is a reasonable simple non-parametric comparison. The p-value is not a measure of practical importance, and this unadjusted test does not control for other variables.

In [ ]:
from scipy.stats import mannwhitneyu

female_visits = data.loc[data["gender"] == "female", "visits"]
male_visits = data.loc[data["gender"] == "male", "visits"]

test = mannwhitneyu(female_visits, male_visits, alternative="two-sided")
print("Mann–Whitney U statistic:", test.statistic)
print("p-value:", test.pvalue)
print("Female mean:", round(female_visits.mean(), 3))
print("Male mean:", round(male_visits.mean(), 3))


The dataset shows a statistically detectable unadjusted difference in the distributions of visit counts by gender (p < 0.001). This does not establish a cause or imply that gender itself explains the difference. A multivariable model and context about the sample would be needed for deeper conclusions.

## 14. Export cleaned data for Power BI

This exports the analytical columns plus two exploratory bands and a visit-status label. The band labels are numeric ranges only until the original data dictionary confirms units.

In [ ]:
powerbi_data = data.copy()
powerbi_data["visit_status"] = np.where(powerbi_data["visits"] > 0, "One or more visits", "Zero visits")

powerbi_data.to_csv("healthcare_doctor_visits_powerbi.csv", index=False)
print("Exported healthcare_doctor_visits_powerbi.csv")
print("Power BI dataset shape:", powerbi_data.shape)


## 15. Power BI dashboard plan

Import `healthcare_doctor_visits_powerbi.csv` into Power BI Desktop.

### Suggested dashboard layout
**Top KPI cards**
- Total records
- Average visits
- % of records with one or more visits
- % of records with zero visits

**Charts**
1. Column chart: average visits by illness count
2. Bar chart: average visits by gender
3. Column chart: average visits by age value band
4. Column chart: average visits by income value band
5. Scatter chart: reduced-activity days vs visits
6. Scatter chart: poor-health days vs visits
7. Slicers: gender, visit status, private, freepoor, freerepat, nchronic, lchronic

### Useful DAX measures
```DAX
Total Records = COUNTROWS('healthcare_doctor_visits_powerbi')

Average Visits = AVERAGE('healthcare_doctor_visits_powerbi'[visits])

Records With Visits =
CALCULATE(
    COUNTROWS('healthcare_doctor_visits_powerbi'),
    'healthcare_doctor_visits_powerbi'[visits] > 0
)

Percent With Visits =
DIVIDE([Records With Visits], [Total Records], 0)

Zero Visit Records =
CALCULATE(
    COUNTROWS('healthcare_doctor_visits_powerbi'),
    'healthcare_doctor_visits_powerbi'[visits] = 0
)

Percent Zero Visits =
DIVIDE([Zero Visit Records], [Total Records], 0)
```

Format percentage measures as percentages in Power BI. Keep dashboard labels neutral and avoid implying causation.

## 16. Final conclusions

1. The dataset has 5,190 records and no missing values or exact duplicate rows.
2. 4,141 records (79.79%) show zero visits; 1,049 (20.21%) show one or more visits.
3. Average visit counts generally rise with reported illness count.
4. Female records have a higher mean visit count than male records in this dataset.
5. Reduced-activity days show the strongest Spearman association with visits among the numeric variables examined.
6. The age and income units, the meaning of binary indicators, survey time period, and sampling method should be verified before making real-world or policy recommendations.

**Project limitation:** This is descriptive analytics on observational data. The results describe patterns in the provided records; they do not prove cause and effect or establish clinical recommendations.